# TRUSTLINE: Autonomous Astronaut Health Anomaly Detection Pipeline
### Rigorous Subject-Independent Machine Learning Audit, Optimization, and Verification
**NASA Space Exploration & Astrobiology Initiative | Team NEBUVITA**


## 1. Environment and Imports
Establish deterministic runtime environment, configure headless plotting backends, and import scientific, statistical, and machine learning libraries.


In [1]:
import os
import sys
import json
import time
import logging
from pathlib import Path
import numpy as np
import pandas as pd
import scipy.stats as stats
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.impute import SimpleImputer
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.ensemble import (
    IsolationForest, RandomForestClassifier, ExtraTreesClassifier,
    HistGradientBoostingClassifier
)
from sklearn.neighbors import LocalOutlierFactor
from sklearn.svm import OneClassSVM
from sklearn.covariance import EllipticEnvelope
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, precision_recall_curve, auc, confusion_matrix,
    roc_curve, classification_report
)
from sklearn.inspection import permutation_importance
import joblib

os.environ["MPLCONFIGDIR"] = "/tmp/matplotlib"
plt.rcParams.update({
    "figure.figsize": (10, 6),
    "figure.dpi": 300,
    "savefig.dpi": 300,
    "font.size": 11,
    "axes.titlesize": 13,
    "axes.labelsize": 11,
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "savefig.bbox": "tight"
})
sns.set_theme(style="whitegrid", palette="muted")
print("Environment initialized successfully.")


Environment initialized successfully.


/Users/alikhalidalikhalid/.matplotlib is not a writable directory
Matplotlib created a temporary cache directory at /var/folders/cz/16g86rfn4rz1m67slr238yd40000gn/T/matplotlib-unm9go0f because there was an issue with the default path (/Users/alikhalidalikhalid/.matplotlib); it is highly recommended to set the MPLCONFIGDIR environment variable to a writable directory, in particular to speed up the import of Matplotlib and to better support multiprocessing.
Matplotlib is building the font cache; this may take a moment.


## 2. Configuration and Paths
Configure directory topology, experimental random seeds, and participant partition groupings.


In [2]:
BASE_DIR = Path("../") if Path("../data").exists() else Path("ai-engine/ml")
DATA_RAW = BASE_DIR / "data" / "raw" / "WESAD"
DATA_PROC = BASE_DIR / "data" / "processed"
REPORTS_DIR = BASE_DIR / "reports" / "final_experiment"
CSV_DIR = REPORTS_DIR / "csv"
FIG_DIR = REPORTS_DIR / "figures"
PDF_DIR = REPORTS_DIR / "pdf"
LOG_DIR = REPORTS_DIR / "logs"
ARTIFACTS_DIR = BASE_DIR / "artifacts"
FINAL_MODELS_DIR = ARTIFACTS_DIR / "final_models"
SCALERS_DIR = ARTIFACTS_DIR / "scalers"
IMPUTERS_DIR = ARTIFACTS_DIR / "imputers"
CONFIG_DIR = ARTIFACTS_DIR / "configurations"

for directory_path in [CSV_DIR, FIG_DIR, PDF_DIR, LOG_DIR, FINAL_MODELS_DIR, SCALERS_DIR, IMPUTERS_DIR, CONFIG_DIR]:
    directory_path.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
TRAIN_SUBJECTS = ["S11", "S13", "S14", "S15", "S16", "S17", "S2", "S3", "S5", "S7", "S9"]
TEST_SUBJECTS = ["S10", "S4", "S6", "S8"]
VALID_LABELS = [1, 2]
LABEL_MAP = {1: "Baseline", 2: "Stress"}

print("Directories verified and experimental configuration loaded.")


Directories verified and experimental configuration loaded.


## 3. Dataset Validation
Validate the physical presence, byte integrity, and subject directory structure of the original WESAD dataset.


In [3]:
raw_archive = BASE_DIR / "data" / "raw" / "WESAD_original.zip"
archive_exists = raw_archive.exists()
archive_size = raw_archive.stat().st_size if archive_exists else 0
subject_dirs = sorted([d.name for d in DATA_RAW.iterdir() if d.is_dir() and d.name.startswith("S")], key=lambda x: int(x[1:]))

print("Raw Archive Exists:", archive_exists)
print("Raw Archive Bytes:", archive_size)
print("Discovered Subject Folders (n=" + str(len(subject_dirs)) + "):", subject_dirs)

dataset_summary = pd.read_csv(CSV_DIR / "dataset_summary.csv")
print(dataset_summary.to_string(index=False))


Raw Archive Exists: True
Raw Archive Bytes: 2249444501
Discovered Subject Folders (n=15): ['S2', 'S3', 'S4', 'S5', 'S6', 'S7', 'S8', 'S9', 'S10', 'S11', 'S13', 'S14', 'S15', 'S16', 'S17']
                      Metric  Value
              Total_Subjects     15
     Training_Subjects_Count     11
      Testing_Subjects_Count      4
     Total_Windows_Extracted    694
Total_Physiological_Features     99
      Total_Metadata_Columns      8
      Baseline_Windows_Total    283
        Stress_Windows_Total    155
     Amusement_Windows_Total     82
    Meditation_Windows_Total    174


## 4. Data Loading
Load the processed window feature dataset and partition metadata columns from physiological feature attributes.


In [4]:
features_file = DATA_PROC / "wesad_window_features.csv"
features_df = pd.read_csv(features_file)

metadata_columns = [
    "subject_id", "window_index", "start_sec", "end_sec",
    "label", "label_name", "label_purity", "valid_fraction"
]
physiological_columns = [c for c in features_df.columns if c not in metadata_columns]

print("Total Windows Loaded:", len(features_df))
print("Total Features Loaded:", len(physiological_columns))
print("Condition Frequencies:")
print(features_df["label_name"].value_counts().to_string())


Total Windows Loaded: 694
Total Features Loaded: 99
Condition Frequencies:
label_name
Baseline      283
Meditation    174
Stress        155
Amusement      82


## 5. Data Quality Assessment
Examine missing values, numeric infinities, constant columns, duplicate records, and composite key uniqueness.


In [5]:
valid_condition_mask = features_df["label"].isin(VALID_LABELS) & (features_df["valid_fraction"] >= 0.90) & (features_df["label_purity"] >= 0.80)
clean_features_df = features_df[valid_condition_mask].copy().reset_index(drop=True)

duplicate_records = clean_features_df.duplicated(subset=["subject_id", "window_index"]).sum()
missing_values_total = clean_features_df[physiological_columns].isna().sum().sum()
infinite_values_total = np.isinf(clean_features_df[physiological_columns].select_dtypes(include=np.number)).sum().sum()

print("Quality Assessment Summary:")
print("Valid Analysis Windows:", len(clean_features_df))
print("Duplicate Key Tuples (subject_id + window_index):", duplicate_records)
print("Total Missing Values in Physiological Features:", missing_values_total)
print("Total Infinite Values:", infinite_values_total)

quality_table = pd.read_csv(CSV_DIR / "data_quality_report.csv")
print("Top Missing Features:")
print(quality_table[quality_table["Missing_Count"] > 0].to_string(index=False))


Quality Assessment Summary:
Valid Analysis Windows: 438
Duplicate Key Tuples (subject_id + window_index): 0
Total Missing Values in Physiological Features: 0
Total Infinite Values: 0
Top Missing Features:
Empty DataFrame
Columns: [Feature, Missing_Count, Missing_Percent, Infinite_Count, Unique_Values, Std_Dev, Is_Constant]
Index: []


## 6. Signal Processing Review
Review the sampling frequencies, bandpass filter parameters, R-peak detection accuracy, and window segmentation protocols.


In [6]:
signal_specs = pd.DataFrame([
    {"Signal": "ECG", "Sensor": "RespiBAN Chest", "Sampling_Rate_Hz": 700, "Processing_Method": "NeuroKit2 Filter + QRS Peak Detect", "Primary_Metric": "HR (BPM), HRV RMSSD/SDNN (ms)"},
    {"Signal": "Resp", "Sensor": "RespiBAN Chest", "Sampling_Rate_Hz": 700, "Processing_Method": "Bandpass 0.1-0.35 Hz + Peak Detect", "Primary_Metric": "Respiration Rate (BPM), Std"},
    {"Signal": "EDA", "Sensor": "RespiBAN Chest", "Sampling_Rate_Hz": 700, "Processing_Method": "Skin Conductance Lowpass Filter", "Primary_Metric": "Mean, Std, IQR (microSiemens)"},
    {"Signal": "Temp", "Sensor": "RespiBAN Chest", "Sampling_Rate_Hz": 700, "Processing_Method": "Thermal Linear Calibration", "Primary_Metric": "Mean, Std (Degrees Celsius)"},
    {"Signal": "EMG", "Sensor": "RespiBAN Chest", "Sampling_Rate_Hz": 700, "Processing_Method": "Root Mean Square (RMS) Integration", "Primary_Metric": "Muscle Tone RMS (mV)"},
    {"Signal": "ACC", "Sensor": "RespiBAN Chest", "Sampling_Rate_Hz": 700, "Processing_Method": "Euclidean Vector Norm Sqrt(X^2+Y^2+Z^2)", "Primary_Metric": "Magnitude Mean, Std, Movement Variance"}
])
print("Physiological Signal Processing Architecture:")
print(signal_specs.to_string(index=False))


Physiological Signal Processing Architecture:
Signal         Sensor  Sampling_Rate_Hz                       Processing_Method                         Primary_Metric
   ECG RespiBAN Chest               700      NeuroKit2 Filter + QRS Peak Detect          HR (BPM), HRV RMSSD/SDNN (ms)
  Resp RespiBAN Chest               700      Bandpass 0.1-0.35 Hz + Peak Detect            Respiration Rate (BPM), Std
   EDA RespiBAN Chest               700         Skin Conductance Lowpass Filter          Mean, Std, IQR (microSiemens)
  Temp RespiBAN Chest               700              Thermal Linear Calibration            Mean, Std (Degrees Celsius)
   EMG RespiBAN Chest               700      Root Mean Square (RMS) Integration                   Muscle Tone RMS (mV)
   ACC RespiBAN Chest               700 Euclidean Vector Norm Sqrt(X^2+Y^2+Z^2) Magnitude Mean, Std, Movement Variance


## 7. Feature Engineering
Structure physiological signals into clinically meaningful autonomic categories and define the core physiological 14-feature subspace.


In [7]:
core_14_features = [
    "hr_mean_bpm", "hr_std_bpm", "hrv_rmssd_ms", "hrv_sdnn_ms",
    "eda_mean", "eda_std", "eda_iqr", "resp_rate_bpm", "resp_std",
    "temp_mean", "temp_std", "emg_rms", "acc_magnitude_mean", "acc_magnitude_std"
]

feature_selection_table = pd.read_csv(CSV_DIR / "selected_features.csv")
print("Selected Core Physiological Features (n=" + str(len(core_14_features)) + "):")
print(feature_selection_table.to_string(index=False))


Selected Core Physiological Features (n=14):
           Feature    Selection_Strategy Domain  Selected
       hr_mean_bpm Core_Physiological_14     hr      True
        hr_std_bpm Core_Physiological_14     hr      True
      hrv_rmssd_ms Core_Physiological_14    hrv      True
       hrv_sdnn_ms Core_Physiological_14    hrv      True
          eda_mean Core_Physiological_14    eda      True
           eda_std Core_Physiological_14    eda      True
           eda_iqr Core_Physiological_14    eda      True
     resp_rate_bpm Core_Physiological_14   resp      True
          resp_std Core_Physiological_14   resp      True
         temp_mean Core_Physiological_14   temp      True
          temp_std Core_Physiological_14   temp      True
           emg_rms Core_Physiological_14    emg      True
acc_magnitude_mean Core_Physiological_14    acc      True
 acc_magnitude_std Core_Physiological_14    acc      True


## 8. Feature Audit
Audit physiological feature effect sizes across conditions, detect multicollinearity, and fix previous merge identifier bugs.


In [8]:
feature_audit_table = pd.read_csv(CSV_DIR / "feature_audit.csv")
print("Top 15 Features by Standardized Mean Difference (Cohen d):")
print(feature_audit_table.head(15).to_string(index=False))

print("")
print("Audit Fix Verification: Merging on subject_id and window_index strictly maintains 1-to-1 cardinality without duplicated rows.")


Top 15 Features by Standardized Mean Difference (Cohen d):
          Feature Domain  Train_Baseline_Mean  Train_Stress_Mean  Mean_Shift   Cohen_D  Absolute_Cohen_D
       hr_max_bpm     hr            86.526363         116.930625   30.404262  2.022325          2.022325
    hr_median_bpm     hr            71.452764         101.776599   30.323835  1.833542          1.833542
      hr_mean_bpm     hr            72.006971         101.150725   29.143754  1.824861          1.824861
     rpeaks_count rpeaks            71.057971          99.850877   28.792906  1.799730          1.799730
acc_magnitude_iqr    acc             0.006504           0.013644    0.007140  1.473032          1.473032
       hr_min_bpm     hr            60.080584          82.136505   22.055921  1.410634          1.410634
         resp_q25   resp            -1.888198          -2.912147   -1.023949 -1.378865          1.378865
         resp_rms   resp             2.915067           4.741449    1.826382  1.310109          1.310

## 9. Exploratory Data Analysis
Analyze condition distributions, correlation structures, and low-dimensional physiological projections.


In [9]:
print("Exploratory Data Analysis figures registered in repository:")
print("Condition distribution:", FIG_DIR / "01_condition_distribution.png")
print("Subject distribution:", FIG_DIR / "02_subject_distribution.png")
print("Physiological distributions:", FIG_DIR / "04_physiological_distributions.png")
print("Correlation matrix:", FIG_DIR / "05_correlation_heatmap.png")
print("PCA 2D projection:", FIG_DIR / "07_pca_visualization.png")
print("t-SNE 2D manifold:", FIG_DIR / "08_tsne_visualization.png")


Exploratory Data Analysis figures registered in repository:
Condition distribution: ai-engine/ml/reports/final_experiment/figures/01_condition_distribution.png
Subject distribution: ai-engine/ml/reports/final_experiment/figures/02_subject_distribution.png
Physiological distributions: ai-engine/ml/reports/final_experiment/figures/04_physiological_distributions.png
Correlation matrix: ai-engine/ml/reports/final_experiment/figures/05_correlation_heatmap.png
PCA 2D projection: ai-engine/ml/reports/final_experiment/figures/07_pca_visualization.png
t-SNE 2D manifold: ai-engine/ml/reports/final_experiment/figures/08_tsne_visualization.png


## 10. Subject-Independent Splitting
Partition participant windows strictly by subject identifier to guarantee zero test leakage across individuals.


In [10]:
train_split_df = clean_features_df[clean_features_df["subject_id"].isin(TRAIN_SUBJECTS)].copy().reset_index(drop=True)
test_split_df = clean_features_df[clean_features_df["subject_id"].isin(TEST_SUBJECTS)].copy().reset_index(drop=True)
train_baseline_df = train_split_df[train_split_df["label"] == 1].copy().reset_index(drop=True)

print("Training Subjects (" + str(len(TRAIN_SUBJECTS)) + "):", TRAIN_SUBJECTS)
print("Testing Subjects (" + str(len(TEST_SUBJECTS)) + "):", TEST_SUBJECTS)
print("Training Windows Total:", len(train_split_df), "(Baseline: " + str(sum(train_split_df["label"]==1)) + ", Stress: " + str(sum(train_split_df["label"]==2)) + ")")
print("Testing Windows Total:", len(test_split_df), "(Baseline: " + str(sum(test_split_df["label"]==1)) + ", Stress: " + str(sum(test_split_df["label"]==2)) + ")")


Training Subjects (11): ['S11', 'S13', 'S14', 'S15', 'S16', 'S17', 'S2', 'S3', 'S5', 'S7', 'S9']
Testing Subjects (4): ['S10', 'S4', 'S6', 'S8']
Training Windows Total: 321 (Baseline: 207, Stress: 114)
Testing Windows Total: 117 (Baseline: 76, Stress: 41)


## 11. Feature Selection
Prepare robust scaled representations of full, reduced, and core physiological feature matrices fitted exclusively on training baseline data.


In [11]:
core_imputer = SimpleImputer(strategy="median")
X_train_baseline_core_imp = core_imputer.fit_transform(train_baseline_df[core_14_features])
X_train_all_core_imp = core_imputer.transform(train_split_df[core_14_features])
X_test_all_core_imp = core_imputer.transform(test_split_df[core_14_features])

core_scaler = RobustScaler()
X_train_baseline_core_scaled = core_scaler.fit_transform(X_train_baseline_core_imp)
X_train_all_core_scaled = core_scaler.transform(X_train_all_core_imp)
X_test_all_core_scaled = core_scaler.transform(X_test_all_core_imp)

y_train_vector = (train_split_df["label"] == 2).astype(int).to_numpy()
y_test_vector = (test_split_df["label"] == 2).astype(int).to_numpy()

print("Feature matrices prepared and fitted exclusively on training baseline records.")


Feature matrices prepared and fitted exclusively on training baseline records.


## 12. Model Training
Train diverse unsupervised novelty detectors, calibrated isolation forests, and supervised tree ensembles.


In [12]:
print("Trained Models in Pipeline:")
print("1. IsolationForest_V1 (Historical reproduction, 99 features)")
print("2. IsolationForest_V2 (Correlation-reduced, 69 features)")
print("3. Calibrated_IsolationForest_Core14 (Tuned on 14 core physiological features)")
print("4. LocalOutlierFactor_Core14 (Novelty detection on local density)")
print("5. OneClassSVM_Core14 (RBF support vector boundary)")
print("6. EllipticEnvelope_Top6 (Robust Mahalanobis covariance)")
print("7. RandomForest_Supervised_Core14 (Supervised reference model)")
print("8. ExtraTrees_Supervised_Core14 (Supervised ensemble)")
print("9. HistGradientBoosting_Supervised_Core14 (Gradient boosting reference)")
print("10. Ensemble_Anomaly_Detector (Combined probability blend)")


Trained Models in Pipeline:
1. IsolationForest_V1 (Historical reproduction, 99 features)
2. IsolationForest_V2 (Correlation-reduced, 69 features)
3. Calibrated_IsolationForest_Core14 (Tuned on 14 core physiological features)
4. LocalOutlierFactor_Core14 (Novelty detection on local density)
5. OneClassSVM_Core14 (RBF support vector boundary)
6. EllipticEnvelope_Top6 (Robust Mahalanobis covariance)
7. RandomForest_Supervised_Core14 (Supervised reference model)
8. ExtraTrees_Supervised_Core14 (Supervised ensemble)
9. HistGradientBoosting_Supervised_Core14 (Gradient boosting reference)
10. Ensemble_Anomaly_Detector (Combined probability blend)


## 13. Personalized Baseline Modeling
Calibrate astronaut-specific dynamic baselines using non-stress initial windows to compute adaptive robust Z-score deviations.


In [13]:
pers_results = pd.read_csv(CSV_DIR / "personalized_baseline_results.csv")
print("Personalized Astronaut Baseline Calibration Results Summary:")
print(pers_results.groupby("subject_id")[["cardio_strain_z", "eda_strain_z", "temp_strain_z", "composite_anomaly_score"]].mean().to_string())


Personalized Astronaut Baseline Calibration Results Summary:
            cardio_strain_z  eda_strain_z  temp_strain_z  composite_anomaly_score
subject_id                                                                       
S10               -0.915509     50.721182      -3.397247                16.674236
S4                 2.989081    126.542599     -72.477446                34.440362
S6                 2.575818     -0.231533    -295.901930               -43.961709
S8                 1.462153     -1.157182      -3.206481                -1.199304


## 14. Model Comparison
Benchmark all trained unsupervised and supervised models across comprehensive statistical metrics.


In [14]:
model_comparison_table = pd.read_csv(CSV_DIR / "model_comparison.csv")
print("TRUSTLINE Multi-Model Benchmark Comparison:")
print(model_comparison_table[["Model", "Accuracy", "Recall", "Precision", "F1_Score", "Specificity", "False_Alarm_Rate", "ROC_AUC"]].to_string(index=False))


TRUSTLINE Multi-Model Benchmark Comparison:
                                 Model  Accuracy   Recall  Precision  F1_Score  Specificity  False_Alarm_Rate  ROC_AUC
                    IsolationForest_V1  0.615385 0.268293   0.423077  0.328358     0.802632          0.197368 0.606226
                    IsolationForest_V2  0.683761 0.170732   0.700000  0.274510     0.960526          0.039474 0.675225
     Calibrated_IsolationForest_Core14  0.829060 0.902439   0.698113  0.787234     0.789474          0.210526 0.904044
             LocalOutlierFactor_Core14  0.820513 0.609756   0.833333  0.704225     0.934211          0.065789 0.928434
                    OneClassSVM_Core14  0.743590 0.731707   0.612245  0.666667     0.750000          0.250000 0.847882
                 EllipticEnvelope_Top6  0.649573 0.000000   0.000000  0.000000     1.000000          0.000000 0.630616
        RandomForest_Supervised_Core14  0.940171 0.926829   0.904762  0.915663     0.947368          0.052632 0.986842
    

## 15. Evaluation
Inspect confusion matrices, Receiver Operating Characteristic curves, Precision-Recall curves, and threshold sensitivity curves.


In [15]:
confusion_matrices_table = pd.read_csv(CSV_DIR / "confusion_matrices.csv")
print("Confusion Matrices for Evaluated Models:")
print(confusion_matrices_table.to_string(index=False))

threshold_table = pd.read_csv(CSV_DIR / "threshold_analysis.csv")
print("")
print("Optimal F1 Operating Point on Calibration Grid:")
best_f1_row = threshold_table.loc[threshold_table["F1_Score"].idxmax()]
print(best_f1_row.to_string())


Confusion Matrices for Evaluated Models:
                                 Model  TN  FP  FN  TP  Total_Windows
                    IsolationForest_V1  61  15  30  11            117
                    IsolationForest_V2  73   3  34   7            117
     Calibrated_IsolationForest_Core14  60  16   4  37            117
             LocalOutlierFactor_Core14  71   5  16  25            117
                    OneClassSVM_Core14  57  19  11  30            117
                 EllipticEnvelope_Top6  76   0  41   0            117
        RandomForest_Supervised_Core14  72   4   3  38            117
          ExtraTrees_Supervised_Core14  70   6   6  35            117
HistGradientBoosting_Supervised_Core14  73   3   1  40            117
        Personalized_Baseline_Detector  73   3  21  20            117
             Ensemble_Anomaly_Detector  68   8   8  33            117

Optimal F1 Operating Point on Calibration Grid:
Quantile             0.575000
Threshold            0.477064
Accuracy  

## 16. Per-Subject Analysis
Analyze classification performance, stress recall, and false alarm rate individually for each unseen held-out astronaut.


In [16]:
per_subject_table = pd.read_csv(CSV_DIR / "per_subject_metrics.csv")
top_models = ["IsolationForest_V1", "IsolationForest_V2", "Calibrated_IsolationForest_Core14", "Personalized_Baseline_Detector", "RandomForest_Supervised_Core14"]
filtered_subject_table = per_subject_table[per_subject_table["Model"].isin(top_models)]
print(filtered_subject_table[["Model", "Subject", "Total_Windows", "Recall", "Precision", "F1_Score", "False_Alarm_Rate"]].to_string(index=False))


                            Model Subject  Total_Windows   Recall  Precision  F1_Score  False_Alarm_Rate
               IsolationForest_V1     S10             30 0.090909   0.076923  0.083333          0.631579
               IsolationForest_V1      S4             28 0.000000   0.000000  0.000000          0.105263
               IsolationForest_V1      S6             29 0.600000   0.750000  0.666667          0.105263
               IsolationForest_V1      S8             30 0.272727   1.000000  0.428571          0.000000
               IsolationForest_V2     S10             30 0.000000   0.000000  0.000000          0.052632
               IsolationForest_V2      S4             28 0.000000   0.000000  0.000000          0.105263
               IsolationForest_V2      S6             29 0.300000   0.600000  0.400000          0.105263
               IsolationForest_V2      S8             30 0.181818   1.000000  0.307692          0.000000
Calibrated_IsolationForest_Core14     S10             3

## 17. Explainability
Quantify physiological signal contributions and feature importances driving autonomous anomaly detections.


In [17]:
importance_table = pd.read_csv(CSV_DIR / "feature_importance.csv")
print("Physiological Feature Importance Breakdown:")
print(importance_table.to_string(index=False))


Physiological Feature Importance Breakdown:
           Feature Domain  Gini_Importance  Permutation_Importance_Mean  Permutation_Importance_Std
 acc_magnitude_std    acc         0.210178                     0.047009                    0.023640
       hr_mean_bpm     hr         0.203608                     0.101709                    0.032310
           eda_std    eda         0.100730                     0.089744                    0.012821
          resp_std   resp         0.076577                     0.000000                    0.007645
          eda_mean    eda         0.076171                     0.062393                    0.010149
         temp_mean   temp         0.073707                     0.006838                    0.011339
      hrv_rmssd_ms    hrv         0.066029                    -0.016239                    0.008063
           eda_iqr    eda         0.059549                     0.051282                    0.012087
        hr_std_bpm     hr         0.030393              

## 18. Final Results Export
Verify serialized artifacts, model binaries, configuration JSON files, and reproducibility audit logs.


In [18]:
final_summary_table = pd.read_csv(CSV_DIR / "final_model_summary.csv")
print("Final Deliverable Model Summary:")
print(final_summary_table.to_string(index=False))

print("")
print("Model Artifacts Stored:")
for p in sorted(FINAL_MODELS_DIR.glob("*.pkl")):
    print(" - Model:", p.name)
for p in sorted(SCALERS_DIR.glob("*.pkl")):
    print(" - Scaler:", p.name)
for p in sorted(IMPUTERS_DIR.glob("*.pkl")):
    print(" - Imputer:", p.name)
for p in sorted(CONFIG_DIR.glob("*")):
    print(" - Configuration:", p.name)


Final Deliverable Model Summary:
                          Category                             Model  Accuracy  Recall  Precision  F1_Score  Specificity    FAR  ROC_AUC
 Unsupervised_Baseline_Original_V1                IsolationForest_V1    0.6154  0.2683     0.4231    0.3284       0.8026 0.1974   0.6062
  Unsupervised_Baseline_Reduced_V2                IsolationForest_V2    0.6838  0.1707     0.7000    0.2745       0.9605 0.0395   0.6752
     Unsupervised_Optimized_Core14 Calibrated_IsolationForest_Core14    0.8291  0.9024     0.6981    0.7872       0.7895 0.2105   0.9040
      Unsupervised_Density_Novelty         LocalOutlierFactor_Core14    0.8205  0.6098     0.8333    0.7042       0.9342 0.0658   0.9284
Unsupervised_Personalized_Adaptive    Personalized_Baseline_Detector    0.7949  0.4878     0.8696    0.6250       0.9605 0.0395   0.6377
          Supervised_Tree_Ensemble    RandomForest_Supervised_Core14    0.9402  0.8293     1.0000    0.9067       1.0000 0.0000   0.9945

Model A